# Setup & Installation

Uncomment the appropriate lines below depending on your environment.

**Local Development:**
Run this in your terminal:
`pip install -e src/dcl_agent`

**Google Colab:**

In [1]:
# !git clone https://github.com/EgorBEremeev/dcl.git
# !pip install -e dcl/src/dcl_agent
# !pip install -e ../src/dcl_agent

In [1]:
import os
from pathlib import Path
import json

In [2]:
from dcl_agent.agent import DCLAgent
from dcl_agent.adapter.gemini import GeminiAdapter
from dcl_agent.adapter.mock import MockLLMAdapter

In [4]:
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    print("Enter your GEMINI API Key:")
    api_key = input().strip()
    if not api_key:
        print("Error: API Key is required.")

In [ ]:
# print(api_key)

In [8]:
print(list(Path().cwd().parents)[1])

c:\git


In [13]:
root_dir_dcl_core = Path().cwd().parents[1] / "dcl-agents" / "dcl-core"
print(root_dir_dcl_core)

c:\git\dcl-agents\dcl-core


In [3]:
def _bundles_to_load():
    # Path to dcl-agent artifacts
    root_dir = Path().cwd().parent
    dcl_core_dir = Path().cwd().parents[1] / "dcl-agents" / "dcl-core"
    bundles = [
        str(dcl_core_dir),
        str(root_dir / "adpilot-core")
    ]

    for b in bundles:
            if not Path(b).exists():
                print(f"Error: Bundle path {b} not found.")
                return
    return bundles

bundles = _bundles_to_load()

print(f"Bundles to load: {bundles}...")

Bundles to load: ['c:\\git\\dcl-agents\\dcl-core', 'c:\\git\\dcl-adpilot\\adpilot-core']...


# AI Studio Simulation
AI Studio можно воспользоваться, когда вы хотите поэксперементировать бесплатно, без необходимости в API-ключе.
Для симуляции можно сделать дамп полной конфигурации dcl-агента и поместить его в System Prompt в AI Studio. Далее в чате писать DLC-инструкции на разработанном диалекте языка. 

Симуляция сопровождается рядом понятных ограничений:
* В режиме чата Invocation Context с каждым сообщением дополняется историей диалога.
* Полная конфигуарция агента, помещаемая в System Prompt, с одной стороны избыточна, с другой стороны дает расширенный контекст для конкретной DCL-инструкции. При вызове через API агент, предполагается, будет формировать специализированный для конкретной инструкции Invocation Context. 
* Также нужно осознавать, что полная конфигуарция агента из System Prompt может быть предобработанна внутренними механизмами AIStudio несколько иначе чем при вызовах через API.

Первые тесты `god-mode` агента показывают работоспособность и продуктивность режима симуляции. Кроме того режим диалога дает поле возможностей для ретроспективы и рефакторинга промпт-модулей на основе истории сессий симуляции.

In [12]:
agent_sim = DCLAgent(bundles=bundles, adapter=MockLLMAdapter())

In [13]:
# Check Registry
registry = agent_sim.get_registry() 

print(json.dumps(registry._aliases, indent=2))

for module in registry.list_modules():
    print(module)

{
  "INFER": "ad-pilot/operations/infer/1.0",
  "DEFINE": "ad-pilot/operations/define/1.6",
  "ENRICH": "ad-pilot/operations/enrich/1.4",
  "EXTRACT": "ad-pilot/operations/extract/1.2",
  "GRAMMAR": "dcl-core/knowledges/framework/dcl_grammar/2.2",
  "SCHEMA": "dcl-core/modifiers/dcl_specification/3.0",
  "LENS_GEO": "modifiers/market/geo/1.0",
  "LENS_EXPERTISE": "modifiers/market/expertise/1.2",
  "LENS_SCENARIO": "modifiers/market/scenario/1.3",
  "LENS_VALUE": "modifiers/market/value/1.0",
  "AXES_MODEL": "knowledges/framework/3-axes_attribute_model/1.0",
  "dclc": "dcl-core/knowledges/ontology/dcl-core.ttl",
  "ctx": "dcl-core/knowledges/ontology/agent_context_onlology.ttl"
}
dcl-core/modifiers/dcl_specification/3.0
dcl-core/knowledges/framework/dcl_grammar/2.2
dcl-core/knowledges/ontology/agent_context_onlology.ttl
dcl-core/knowledges/ontology/dcl-core.ttl
ad-pilot/operations/define/1.6
ad-pilot/operations/define/1.4
ad-pilot/operations/enrich/1.4
ad-pilot/operations/enrich/1.3
ad

In [15]:
cmd_sim_full_config = """
INFER Context_Fingerprint
FROM "Draft_Record - Пользовательский текст. Короткое, хаотичное описание, товара. Возможно с некоторым набором атрибутов и значений в yaml-списках. Пример - скейт-борд для детей."
USING INFER, DEFINE, ENRICH, EXTRACT, LENS_GEO, LENS_EXPERTISE, LENS_SCENARIO, LENS_VALUE, AXES_MODEL, GRAMMAR, SCHEMA, dclc, ctx
"""
print(f"\nExecuting Command: {cmd_sim_full_config}")

instruction_sim = agent_sim.parser.parse(cmd_sim_full_config)
print("\n--- Parsed Instruction ---")
print(instruction_sim)
print("-----------------------")
invocation_context_sim = agent_sim.strategy.assemble(instruction_sim, agent_sim.registry)
print("\n--- Invocation Context ---")
for i, frame in enumerate(invocation_context_sim.frames[:]):
    print(f'Frame {i}:')
    print(frame.content, "\n")
print("-----------------------")


Executing Command: 
INFER Context_Fingerprint
FROM "Draft_Record - Пользовательский текст. Короткое, хаотичное описание, товара. Возможно с некоторым набором атрибутов и значений в yaml-списках. Пример - скейт-борд для детей."
USING INFER, DEFINE, ENRICH, EXTRACT, LENS_GEO, LENS_EXPERTISE, LENS_SCENARIO, LENS_VALUE, AXES_MODEL, GRAMMAR, SCHEMA, dclc, ctx


--- Parsed Instruction ---
Instruction(action='INFER', operand=Entity(type='ANY', value='Context_Fingerprint'), sources=[ResourceRef(id='Draft_Record - Пользовательский текст. Короткое, хаотичное описание, товара. Возможно с некоторым набором атрибутов и значений в yaml-списках. Пример - скейт-борд для детей.', type=None)], modifiers=[ResourceRef(id='INFER', type=None), ResourceRef(id='DEFINE', type=None), ResourceRef(id='ENRICH', type=None), ResourceRef(id='EXTRACT', type=None), ResourceRef(id='LENS_GEO', type=None), ResourceRef(id='LENS_EXPERTISE', type=None), ResourceRef(id='LENS_SCENARIO', type=None), ResourceRef(id='LENS_VALUE'

# Агент с Gemini API

Требуется API-ключ

In [36]:
# Use Gmini Adapter
try:
    # adapter = GeminiAdapter(api_key=api_key, model_name="gemini-2.flash-exp")
    # Модели доступные на Free Tier по текущим лимитам, которые можно смотреть в https://aistudio.google.com/usage?project=gen-lang-client-0013706841&timeRange=last-28-days&tab=rate-limit
    # gemini-3-flash-preview
    # gemini-2.5-flash
    # gemini-2.5-flash-lite
    adapter = GeminiAdapter(api_key=api_key, model_name="models/gemini-3-flash-preview")
except ImportError:
    print("Error: google-genai not installed. Run `pip install google-genai`.")
except Exception as e:
    print(f"Error initializing adapter: {e}")

In [42]:
agent = DCLAgent(bundles=bundles, adapter=adapter)

In [ ]:
# Можно проверить доступне для ключа модели и их точные названя. Полезно для Free-Tier ключа, можно выбирать доступные по лимитам модели.
list(agent.adapter.client.models.list())

[Model(
   description='Obtain a distributed representation of a text.',
   display_name='Embedding Gecko',
   input_token_limit=1024,
   name='models/embedding-gecko-001',
   output_token_limit=1,
   supported_actions=[
     'embedText',
     'countTextTokens',
   ],
   tuned_model_info=TunedModelInfo(),
   version='001'
 ),
 Model(
   description='Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.',
   display_name='Gemini 2.5 Flash',
   input_token_limit=1048576,
   max_temperature=2.0,
   name='models/gemini-2.5-flash',
   output_token_limit=65536,
   supported_actions=[
     'generateContent',
     'countTokens',
     'createCachedContent',
     'batchGenerateContent',
   ],
   temperature=1.0,
   thinking=True,
   top_k=64,
   top_p=0.95,
   tuned_model_info=TunedModelInfo(),
   version='001'
 ),
 Model(
   description='Stable release (June 17th, 2025) of Gemini 2.5 Pro',
   display_name='Gemini 2.5 Pr

In [43]:
# Check Registry
registry = agent.get_registry() 

print(json.dumps(registry._aliases, indent=2))

for module in registry.list_modules():
    print(module)
    


{
  "PromptModule": "dcl-god-mode/entities/prompt_module/1.1",
  "WRITE": "dcl-god-mode/operations/write/4.0",
  "REFINE": "dcl-god-mode/operations/refine/8.1",
  "DECOMPOSE": "dcl-god-mode/operations/decompose/1.0",
  "GRAMMAR": "dcl-core/knowledges/framework/dcl_grammar/2.2",
  "SCHEMA": "dcl-core/modifiers/dcl_specification/3.0",
  "dclc": "dcl-core/knowledges/ontology/dcl-core.ttl",
  "ctx": "dcl-core/knowledges/ontology/agent_context_onlology.ttl",
  "dclgm": "dcl-god-mode/knowledges/ontology/dcl-god-mode.ttl"
}
dcl-core/modifiers/dcl_specification/3.0
dcl-core/knowledges/framework/dcl_grammar/2.2
dcl-core/knowledges/ontology/agent_context_onlology.ttl
dcl-core/knowledges/ontology/dcl-core.ttl
dcl-god-mode/entities/prompt_module/1.1
sys/lenses/component_arch_v2/2.0
dcl-god-mode/modifiers/mod_prompt_patterns_collection.md
sys/lenses/w3c_bnf_grammar/1.1
sys/lenses/w3c_owl_ontology/1.1
sys/lenses/w3c_schema_spec/1.1
dcl-god-mode/operations/decompose/1.0
dcl-god-mode/operations/refine

In [10]:
cmd = """
WRITE PromptModule('dcl-god-mode/modifiers/poetry.yaml')
USING knowledges/framework/dcl_god_mode/0.2, GRAMMAR, SCHEMA, dclc, ctx, 'dclgm'
"""

In [ ]:
# ВАЖНО!!! После исполнения запроса закоментировать, чтобы случайно при запуске всего ноутбука не СЖЕЧЬ все ЛИМИТЫ!!!!
#  response = agent.execute(cmd)
print("\n--- Gemini Response ---")
# print(response)
print("-----------------------")


--- Gemini Response ---
Принято. Выполнение операции `WRITE` для сущности 'PROMPT_MODULE'.

**Исполнитель:** Агент-Методолог (использующий логику `dcl-god-mode/operations/write`).
**Интенция:** Создать модуль-Модификатор (`MODIFIER`), который будет накладывать требования к генерации текстовых ответов в поэтической форме.
**Линзы:** DCL Domain Ontology and Specification (из `dcl-core/modifiers/dcl_specification`), Базовая онтология DCL (из `dclc`), Онтология контекста агента (из `ctx`), Онтология DCL God Mode (из `dclgm`).

---

### Артефакт: `dcl-god-mode/modifiers/poetry.yaml`

```yaml
# METADATA
id: dcl-god-mode/modifiers/poetry.yaml
type: MODIFIER
version: 1.0
description: "Модификатор, который накладывает стилистические и формальные ограничения на текстовую генерацию, заставляя агента отвечать в поэтической форме, соблюдая рифму и ритм."

# IDENTITY
role: "Поэтический Стилист"
worldview: "Вы — душа поэта, облекающая мысли в рифмы и ритмы. Ваша задача — преобразить прозу в изящные 